# 🐦‍⬛ Raven Nano — Training Pipeline

10M parameter transformer LLM trained from scratch.

**Execution order:**
1. Mount Drive → Install deps
2. Upload code files (.py) + corpus
3. Train tokenizer → Prepare data
4. Train model → Test inference

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/raven_nano/checkpoints', exist_ok=True)
print('Drive mounted. Checkpoint directory ready.')

## 2. Install Dependencies

In [ ]:
!pip install tokenizers --quiet
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 3. Upload Code Files & Corpus
Upload all `.py` files (`config.py`, `tokenizer_train.py`, `data_prep.py`, `model.py`, `train.py`, `inference.py`) **and** your `corpus.txt` in one go.

In [ ]:
import os
from google.colab import files

# Create project structure
PROJECT_DIR = '/content/raven_nano'
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
%cd {PROJECT_DIR}

# Upload all files at once
print('Upload all .py files AND corpus.txt together:')
uploaded = files.upload()

for filename, data in uploaded.items():
    if filename.endswith('.py'):
        # Python files go in project root
        dest = f'{PROJECT_DIR}/{filename}'
    elif filename == 'corpus.txt':
        # Corpus goes in data/
        dest = f'{PROJECT_DIR}/data/{filename}'
    else:
        dest = f'{PROJECT_DIR}/{filename}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  ✓ {filename} → {dest}')

print(f'\n📁 Project structure:')
for root, dirs, fnames in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for fn in fnames:
        print(f'{indent}  {fn}')

## 4. Train Tokenizer

In [ ]:
!python tokenizer_train.py

## 5. Prepare Data

In [ ]:
!python data_prep.py

## 6. Train (Fresh Start)

In [ ]:
!python train.py

## 6b. Resume Training (if disconnected)

In [ ]:
!python train.py --resume

## 7. Check Training Progress

In [ ]:
import torch
ckpt = torch.load('checkpoints/raven_nano_best.pt', map_location='cpu')
print(f'Best checkpoint:')
print(f'  Step     : {ckpt["step"]}')
print(f'  Val loss : {ckpt["val_loss"]:.4f}')

## 8. Test Inference

In [ ]:
import sys
sys.path.insert(0, '.')

from inference import load_model_for_inference, sample
from tokenizers import Tokenizer
from config import cfg

tok   = Tokenizer.from_file(cfg.TOKENIZER_PATH)
model = load_model_for_inference()

test_inputs = [
    'hi',
    "i can't study tonight",
    'gonna hit the gym',
    "i'm a failure",
    'what is 47 times 83',
    'good morning',
    "i'll start tomorrow",
]

print('=' * 50)
for inp in test_inputs:
    response = sample(model, tok, inp)
    print(f'User  : {inp}')
    print(f'Raven : {response}')
    print()